In [7]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RESULTS_DIR, SRC_DIR, SRC_DIRS, CHOC_SCENARIO

CONTENT = "dir"        # "dir" : rd_dir_CI | "tot" : rd_tot_CI
YEAR = 2022 # YEARS[0]       # year of the base table (the same for all the years of YEARS_CALC)
YEARS_CALC = list(range(2025, 2051))                   # years computed: the shock of year t is the level gap vs the baseline of year t (nothing to cumulate)
DISPLAY_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]   # years displayed in the table (among YEARS_CALC); YEAR = base year, variation 0
OMEGA_FILE = "omega_progressif.csv"                               # inputs/: origin, dest, sector, delta_pct, + year column for one ω per year (without a year column: the same ω every year)

EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
        "RO", "SK", "SI", "ES", "SE"]
BLOCKS = {
    "USA": ["US"],
    "EU27": EU27,
    "RestOECD": ["AU", "CA", "JP", "KR", "MX", "NZ", "NO", "CH", "UK"],
    "China": ["CN"],
    "RestBRICS": ["BR", "RU", "IN", "ZA"],
    "RestAsia": ["BD", "KH", "ID", "LA", "MY", "MM", "NP", "PK", "PH", "SG", "TW", "TJ", "TL", "TH", "VN"],
}
REGION_TO_BLOCK = {region: block for block, regions in BLOCKS.items() for region in regions}
DEFAULT_BLOCK = "RoW"

MANUFACTURING = ["C13_C15", "C16_C18", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
NONMANUF = "NONMANUF"
AGGREGATES = ["C", "H"]    # sector groups (prefix of the sector codes) also computed as one aggregate sector, as add_sector() of the 2.0

# ---- final file in IAMC format: Model | Scenario | Region | Variable | Unit | one column per year -----------------------------------
# Bilateral version: Region = bloc whose sector's unit cost changes, Variable = "<sector> - <bloc that supplies the inputs>"
IAMC_MODEL = "NEMESIS-World v1.1"                                     # "Model" column
IAMC_SCENARIO = "model{model}_case{scenario}"  # "Scenario" column: {model} = 2 / 3 / 4 (model of blocks), {scenario} = A / B
IAMC_VARIABLE = "{sector} | {origin}"                        # "Variable" column: {sector} = sector whose unit cost changes ("C" = all manufacturing), {origin} = bloc supplying the inputs
# English labels for the final export only. Internal calculations retain the sector codes.
# Source: NACE Rev. 2, with project-specific combined sectors.
# https://ec.europa.eu/eurostat/fr/web/nace
# https://dd.eionet.europa.eu/vocabulary/eurostat/nace_r2/view
# H is the project aggregate of every code starting with H: it includes H53+J,
# hence information and communication services as well as transportation.
SECTOR_NAMES = {'A': 'Agriculture, forestry and fishing',
 'B': 'Mining and quarrying',
 'C': 'Manufacturing',
 'C10_C12': 'Food, beverages and tobacco',
 'C13_C15': 'Textiles, wearing apparel and leather products',
 'C16_C18': 'Wood, paper and printing products',
 'C19': 'Coke and refined petroleum products',
 'C20+C21': 'Chemicals and pharmaceuticals',
 'C22': 'Rubber and plastic products',
 'C23': 'Other non-metallic mineral products',
 'C24': 'Basic metals',
 'C25': 'Fabricated metal products',
 'C26': 'Computer, electronic and optical products',
 'C27': 'Electrical equipment',
 'C28': 'Machinery and equipment n.e.c.',
 'C29': 'Motor vehicles, trailers and semi-trailers',
 'C30': 'Other transport equipment',
 'C31_C33': 'Furniture, other manufacturing, machinery repair and installation',
 'D': 'Electricity, gas, steam and air conditioning supply',
 'E': 'Water supply, sewerage, waste management and remediation',
 'F': 'Construction',
 'G': 'Wholesale and retail trade, motor vehicle repair',
 'H': 'Transportation, storage, information and communication services',
 'H49': 'Land transport and transport via pipelines',
 'H50': 'Water transport',
 'H51': 'Air transport',
 'H52': 'Warehousing and support activities for transportation',
 'H53+J': 'Postal, courier, information and communication services',
 'I': 'Accommodation and food service activities',
 'K': 'Financial and insurance activities',
 'L': 'Real estate activities',
 'M+N': 'Professional, scientific, technical, administrative and support services',
 'O+U': 'Public administration, defence, social security and extraterritorial organisations',
 'P': 'Education',
 'Q': 'Human health and social work activities',
 'R+S+T': 'Arts, entertainment, recreation, other services and household activities'}

IAMC_YEARS = YEARS_CALC                                       # years written, one column each (among YEARS_CALC)
IAMC_CSV = dict(sep=";", decimal=",")                         # csv format (";" and "," = French Excel opens it directly)
IAMC_XLSX = True                                              # also write the same table as .xlsx

In [8]:
def read_matrix(path):
    return pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])


def flat(m):
    m = m.copy()
    m.index, m.columns = m.index.droplevel(1), m.columns.droplevel(1)
    m.index.names = m.columns.names = ["Region", "Sector"]
    return m


MODEL_BLOCKS = {
    2: {"D": (None, True), "F": (None, False)},
    3: {"I": (True, None), "X": (False, None)},
    4: {"same_same": (True, True), "same_diff": (True, False), "diff_same": (False, True), "diff_diff": (False, False)},
}


def block_matrices(m, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    reg_i, reg_j = m.index.get_level_values("Region"), m.columns.get_level_values("Region")
    foreign = np.asarray(reg_i)[:, None] != np.asarray(reg_j)[None, :]
    rows = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_i], m.index.get_level_values("Sector")], names=["Bloc", "Sector"])
    cols = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_j], m.columns.get_level_values("Sector")], names=["Bloc", "Sector"])
    out = []
    for mask in (~foreign, foreign):
        a = pd.DataFrame(m.to_numpy() * mask, index=rows, columns=cols)
        out.append(a.groupby(level=["Bloc", "Sector"]).sum().T.groupby(level=["Bloc", "Sector"]).sum().T)
    return out


def omega_blocks(index, columns, omega, manufacturing=MANUFACTURING):
    b_i, sec_i = np.asarray(index.get_level_values("Bloc")), np.asarray(index.get_level_values("Sector"))
    b_j = np.asarray(columns.get_level_values("Bloc"))
    om = np.ones((len(index), len(columns)))
    for r in omega.sort_values("sector", key=lambda s: s != "ALL").itertuples():
        rows = (b_i == r.origin) & ((sec_i == r.sector) | (r.sector == "ALL"))
        om[np.ix_(rows, b_j == r.dest)] = 1 + r.delta_pct / 100
    return np.where(np.isin(sec_i, manufacturing)[:, None], om, 1.0)


def shock_bilateral(cd, cf, zd, zf, omega, beta, model, aggregates=AGGREGATES, manufacturing=MANUFACTURING):
    """Same cost shock as shock_blocks + add_sector, but BY BLOC OF ORIGIN of the inputs: dlnCost of the sector x of the bloc y due to what it buys
    from the bloc z. One row per (scenario, bloc y, sector x, origin z).
    - The change of the stock S_k of a buyer is a sum over the supplying blocs z: dS_k(z) = sum over the suppliers i of the bloc z of
      c_ij * (omega_ij - 1) (same masks k as shock_blocks). Scenario B adds the substitution gain g (domestic production fills the gap), attributed
      to the origin z = bloc of the buyer itself.
    - dlnS_k = log((S0 + sum_z dS_k(z)) / S0) is not additive: it is split between the origins in proportion to their share of the stock change,
      dlnS_k(z) = dlnS_k * dS_k(z) / sum_z dS_k(z), so that the sum over z gives back the dlnS_k of shock_blocks exactly.
    - dlnCost(z) = sum_k beta_k * dlnS_k(z): the sum over z is the dlnCost of shock_blocks + add_sector.
    - aggregates (e.g. "C", "H"): the stocks and the changes of the sectors of the group are summed first (as add_sector), then dlnS and the split."""
    idx, cols = cd.index, cd.columns
    bloc_i, sec_i = np.asarray(idx.get_level_values("Bloc")), np.asarray(idx.get_level_values("Sector"))
    bloc_j, sec_j = np.asarray(cols.get_level_values("Bloc")), np.asarray(cols.get_level_values("Sector"))
    origins = np.array(list(dict.fromkeys(bloc_i)))
    n_o, n_d = len(origins), len(cols)
    members = (origins[:, None] == bloc_i[None, :]).astype(float)        # (origin, supplier): the supplier belongs to the bloc
    own = origins[:, None] == bloc_j[None, :]                              # (origin, buyer): the origin is the bloc of the buyer
    same_sec = sec_i[:, None] == sec_j[None, :]
    om = omega_blocks(idx, cols, omega, manufacturing)
    cdv, cfv = cd.to_numpy(), cf.to_numpy()
    types = MODEL_BLOCKS[model]

    s0, shock = {}, {}
    for k, (sec, cty) in types.items():
        ms = np.ones_like(same_sec) if sec is None else same_sec == sec
        d = (cdv * ms).sum(0) if cty is not False else 0.0                  # domestic part (never shocked)
        f0 = (cfv * ms).sum(0) if cty is not True else 0.0
        s0[k] = d + f0
        shock[k] = members @ (cfv * (om - 1.0) * ms) if cty is not True else np.zeros((n_o, n_d))   # change of S_k by origin bloc

    none = np.zeros(n_d)
    cases = [("A", none, none)]
    if zd is not None:                 # scenario B: needs the volumes z
        by_sector = lambda a: pd.DataFrame(a, index=idx, columns=cols).groupby(level="Sector").sum()
        c_dom, z_dom = by_sector(cdv), by_sector(zd.to_numpy())
        lost = by_sector((1 - om) * zf.to_numpy())
        rho_d = (c_dom / z_dom.where(z_dom > 0)).fillna(0.0)
        gain = np.maximum(rho_d * lost, -c_dom)
        g_same = gain.to_numpy()[gain.index.get_indexer(sec_j), np.arange(len(sec_j))]
        cases.append(("B", g_same, gain.sum().to_numpy() - g_same))

    def split(S0, dS):
        """S0: stock before; dS: change of the stock by origin (array over the origins). Returns dlnS by origin."""
        tot = dS.sum(0)
        with np.errstate(all="ignore"):
            dln = np.where(S0 > 0, np.log((S0 + tot) / S0), 0.0)
            share = np.where(tot != 0, dS / np.where(tot != 0, tot, 1.0), 0.0)
        return dln * share

    out = []
    for scenario, g_same, g_diff in cases:
        dS = {}
        for k, (sec, cty) in types.items():
            g = 0.0 if cty is False else (g_same + g_diff if sec is None else g_same if sec else g_diff)
            dS[k] = shock[k] + own * g                                       # the gain g is attributed to the bloc of the buyer
        res = pd.DataFrame({"bloc": np.tile(bloc_j, n_o), "sector": np.tile(sec_j, n_o), "origin": np.repeat(origins, n_d)})
        for k in types:
            res[f"dlnS_{k}"] = split(s0[k], dS[k]).ravel()
        parts = [res]
        for a in aggregates:
            in_a = np.array([s.startswith(a) for s in sec_j])
            for b in dict.fromkeys(bloc_j):
                sel = in_a & (bloc_j == b)
                if not sel.any():
                    continue
                row = pd.DataFrame({"bloc": b, "sector": a, "origin": origins})
                for k in types:
                    S0 = s0[k][sel].sum()
                    dS_a = (dS[k][:, sel] * (s0[k][sel] > 0)).sum(1)          # stocks without a baseline stay at 0, as add_sector
                    row[f"dlnS_{k}"] = split(S0, dS_a)
                parts.append(row)
        out.append(pd.concat(parts, ignore_index=True).assign(scenario=scenario))
    res = pd.concat(out, ignore_index=True)
    if beta is not None:
        res = res.merge(beta, on="sector", how="left")
        res["dlnCost"] = sum(res[f"beta_{k}"] * res[f"dlnS_{k}"] for k in types)
    return res

In [9]:
if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # year whose z_CI is used to rebuild the volumes of the years without z_CI (fixed technical coefficients)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, PATH_WRK + "SRDV (c,s)/", PATH_WRK + "SRDQ (c,s)/"):   # z_CI of the reference year, in any run folder
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes with the technical coefficients of Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D of the supplier per euro sold this year
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "choc_tfp" / "inputs"
omega_all = pd.read_csv(inputs / OMEGA_FILE)                                      # ω of all the years (year column optional)

prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if CHOC_SCENARIO == "B":                      # only scenario B needs the volumes z (A does not)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "scénario A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # domestic / foreign content, aggregated by (block, sector) BEFORE the shock
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "choc_tfp"
out.mkdir(parents=True, exist_ok=True)


volumes 2022 : z_CI exact
238 (bloc, secteur) fournisseurs x 238 (bloc, secteur) acheteurs


In [10]:
def omega_of(year):
    if "year" not in omega_all.columns:
        return omega_all
    om = omega_all[omega_all["year"] == year].drop(columns="year")
    if om.empty:
        raise ValueError(f"{OMEGA_FILE} : aucune ligne pour l'année {year}")
    return om


results = {}
for model in MODEL_BLOCKS:
    f_beta = inputs / ("beta.csv" if model == 2 else f"beta_modele{model}.csv")
    beta_m = pd.read_csv(f_beta) if f_beta.exists() else None
    cache, frames = {}, []
    for year in YEARS_CALC:
        omega = omega_of(year)
        key = omega.to_csv(index=False)
        if key not in cache:
            cache[key] = shock_bilateral(cd, cf, zd, zf, omega, beta_m, model)
        frames.append(cache[key].assign(year=year))
    results[model] = pd.concat(frames, ignore_index=True)
    results[model].to_csv(out / f"resultats{model}_blocs_bilateral_{BASE}_{CONTENT}_horizon_choc{CHOC_SCENARIO}.csv", index=False)
    print(f"modèle ({model}) : {len(YEARS_CALC)} années, {len(results[model])} lignes (bloc, secteur, bloc d'origine) écrites dans {out}")

modèle (2) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp
modèle (3) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp
modèle (4) : 26 années, 91728 lignes (bloc, secteur, bloc d'origine) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp


In [11]:
# Table of the displayed years: one row per (scenario, sector, bloc, origin bloc), one column per year. Change in unit cost of the sector of the bloc
# due to what it buys from the origin bloc, relative to the baseline (the base year does not move: 0)
tables = {}
for model in MODEL_BLOCKS:
    if "dlnCost" not in results[model].columns:
        continue
    wide = results[model].pivot_table(index=["scenario", "sector", "bloc", "origin"], columns="year", values="dlnCost").reindex(columns=DISPLAY_YEARS)
    wide.insert(0, YEAR, 0.0)
    wide = (np.exp(wide) - 1) * 100
    wide.columns.name = None
    tables[model] = wide.round(4).reset_index()
    path = out / f"variation_cout_bilateral_{BASE}_{CONTENT}_modele{model}_base{YEAR}.csv"
    tables[model].to_csv(path, index=False)
    print(f"modèle ({model}) : tableau de {len(tables[model])} lignes écrit dans {path}")

tables[2]

modèle (2) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele2_base2022.csv
modèle (3) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele3_base2022.csv
modèle (4) : tableau de 3528 lignes écrit dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\variation_cout_bilateral_GTAP_dir_modele4_base2022.csv


,scenario,sector,bloc,origin,2022,2025,2030,2035,2040,2045,2050
0,A,A,China,China,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,A,A,China,EU27,0.0,0.0192,0.1165,0.2157,0.3168,0.4199,0.5250
2,A,A,China,RestAsia,0.0,0.0012,0.0070,0.0130,0.0190,0.0252,0.0315
3,A,A,China,RestBRICS,0.0,0.0005,0.0029,0.0054,0.0080,0.0106,0.0132
4,A,A,China,RestOECD,0.0,0.0472,0.2860,0.5298,0.7786,1.0328,1.2925
...,...,...,...,...,...,...,...,...,...,...,...
3523,B,R+S+T,USA,RestAsia,0.0,0.0001,0.0005,0.0008,0.0012,0.0016,0.0020
3524,B,R+S+T,USA,RestBRICS,0.0,0.0002,0.0010,0.0019,0.0028,0.0037,0.0047
3525,B,R+S+T,USA,RestOECD,0.0,0.0076,0.0456,0.0842,0.1233,0.1628,0.2030
3526,B,R+S+T,USA,RoW,0.0,0.0003,0.0018,0.0034,0.0050,0.0066,0.0082


In [12]:
# Final file in IAMC format (same layout as the results of the other models): one row per (Scenario, Region, Variable), one column per year.
# Region = bloc of countries (USA, EU27, China, ...) whose sector's unit cost changes; Variable = "<sector> - <bloc that supplies the inputs>".
# The sum over the origin blocs of one (Region, sector) is the total change of cost of this sector (the own bloc carries the domestic substitution
# of scenario B).
from datetime import date

IAMC_UNIT = "%"


def to_iamc(results, years=DISPLAY_YEARS):
    frames = []
    for model, res in results.items():
        if "dlnCost" not in res.columns:                       # no beta file for this model: no cost change to export
            print(f"modèle ({model}) : pas de dlnCost (fichier beta absent), ignoré")
            continue
        long = res.loc[res["year"].isin(years), ["scenario", "bloc", "sector", "origin", "year", "dlnCost"]].dropna(subset=["dlnCost"]).copy()
        long["value"] = (np.exp(long["dlnCost"]) - 1) * 100
        long["Case"] = [IAMC_SCENARIO.format(model=model, scenario=s) for s in long["scenario"]]
        long["Region (importer)"] = long["bloc"]
        long["Variable (importing sector | exporting region)"] = [
            IAMC_VARIABLE.format(sector=SECTOR_NAMES.get(s, s), origin=o)
            for s, o in zip(long["sector"], long["origin"])
        ]
        frames.append(long[["Case", "Region (importer)", "Variable (importing sector | exporting region)", "year", "value"]])
    long = pd.concat(frames, ignore_index=True)
    wide = long.pivot(index=["Case", "Region (importer)", "Variable (importing sector | exporting region)"], columns="year", values="value").reset_index()
    wide.columns.name = None
    wide.insert(0, "Model", IAMC_MODEL)
    wide.insert(4, "Unit", IAMC_UNIT)
    return wide


iamc = to_iamc(results)
stem = f"ENTICE_T34_{BASE}-{CONTENT}_AllCases_bilateral"
iamc.to_csv(out / f"{stem}.csv", index=False, encoding="utf-8-sig", **IAMC_CSV)
print(f"{len(iamc)} lignes ({iamc['Case'].nunique()} scénarios, {iamc['Region (importer)'].nunique()} régions, {iamc['Variable (importing sector | exporting region)'].nunique()} variables) écrites dans {out / stem}.csv")
iamc.head(10)

10584 lignes (6 scénarios, 7 régions, 252 variables) écrites dans D:\ENTICE\runs\SRDQ (c,s)\choc_tfp\ENTICE_T34_GTAP-dir_AllCases_bilateral.csv


,Model,Case,Region (importer),Variable (importing sector | exporting region),Unit,2025,2030,2035,2040,2045,2050
0,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | China,%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | EU27,%,0.004549,0.027432,0.050543,0.073884,0.097463,0.121283
2,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.000242,0.001458,0.002687,0.003927,0.005179,0.006444
3,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.000119,0.000715,0.001317,0.001925,0.002540,0.003160
4,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | Re...,%,0.014730,0.088841,0.163727,0.239399,0.315883,0.393191
5,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | RoW,%,0.000178,0.001076,0.001982,0.002897,0.003821,0.004754
6,NEMESIS-World v1.1,model2_caseA,China,Accommodation and food service activities | USA,%,0.003004,0.018116,0.033376,0.048787,0.064354,0.080079
7,NEMESIS-World v1.1,model2_caseA,China,"Agriculture, forestry and fishing | China",%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,NEMESIS-World v1.1,model2_caseA,China,"Agriculture, forestry and fishing | EU27",%,0.019242,0.116541,0.215697,0.316776,0.419862,0.525029
9,NEMESIS-World v1.1,model2_caseA,China,"Agriculture, forestry and fishing | RestAsia",%,0.001158,0.007011,0.012969,0.019038,0.025221,0.031523
